# Task 1

In [ ]:
!git clone https://github.com/shankar-veludandi/char-rnn.pytorch.git
%cd char-rnn.pytorch

In [ ]:
!mkdir -p data
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O data/tinyshakespeare.txt

In [ ]:
!pip install unidecode

## RNN

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model rnn \
  --n_epochs 5 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_rnn_5.pt

!python generate.py tinyshakespeare_rnn_5.pt \
  --model rnn

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model rnn \
  --n_epochs 50 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_rnn_50.pt

!python generate.py tinyshakespeare_rnn_50.pt

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model rnn \
  --n_epochs 500 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_rnn_500.pt

!python generate.py tinyshakespeare_rnn_500.pt

The 5‑epoch text is mostly gibberish, with scattered real words and many nonsensical fragments. By 50 epochs, there are more recognizable words, rudimentary sentence structure, and occasional snippets that resemble Shakespearean style, but there are still grammatical errors and odd phrases. By 500 epochs, the text is far more coherent: it maintains line breaks resembling dialogue or verse, uses more plausible vocabulary, and occasionally forms sentences that capture the feel of Elizabethan language. Essentially, each jump in epoch count gives the model more time to learn patterns from the data, leading to less random output and more recognizable structure and vocabulary.

## LSTM

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model lstm \
  --n_epochs 5 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_lstm_5.pt

!python generate.py tinyshakespeare_lstm_5.pt

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model lstm \
  --n_epochs 50 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_lstm_50.pt

!python generate.py tinyshakespeare_lstm_50.pt

In [ ]:
!python train.py data/tinyshakespeare.txt \
  --model lstm \
  --n_epochs 500 \
  --hidden_size 128 \
  --n_layers 2

!mv tinyshakespeare.pt tinyshakespeare_lstm_500.pt

!python generate.py tinyshakespeare_lstm_500.pt

Comparing outputs from the RNN vs. the LSTM model reveals that the LSTM typically produces more coherent text at fewer epochs and continues to improve more noticeably by 500 epochs. With the basic RNN, the text remains relatively garbled longer and struggles to form consistent words or sentence-like structures. The LSTM, by contrast, better retains long-range context and thus tends to learn recognizable vocabulary and Shakespearean style faster. However, both models show only partial coherence at 5 epochs and more fluent text at 500 epochs—an expected result since both are character-level approaches learning from the same data. The key difference is that the LSTM’s gating mechanism allows it to capture dependencies over extended sequences, leading to fewer nonsensical strings and more convincing language than the RNN at comparable epochs.

# Task 2

## Part 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# Load the daily minimum temperatures dataset
df = pd.read_csv("/content/drive/MyDrive/CSCI_6967/HW4/temp_data.csv")

# Sort by Date just in case
df = df.sort_values("Date").reset_index(drop=True)

print("Data columns:", df.columns.tolist())
print(df.head())

# Basic data visualization: Plot entire time series
dates = pd.to_datetime(df["Date"])
temps = df["Temp"].values.astype(np.float32)

plt.figure(figsize=(10,4))
plt.title("Daily Minimum Temperatures in Melbourne (Entire Series)")
plt.plot(dates, temps, label="Temp")
plt.xlabel("Date")
plt.ylabel("Temperature (C)")
plt.legend()
plt.show()

# Create a sliding-window dataset for time-series forecasting
# Predict the next day's temperature from the previous seq_length days.
class TempForecastDataset(Dataset):
    def __init__(self, data_array, seq_length=30):
        super().__init__()
        self.data = data_array
        self.seq_length = seq_length
        self.end = len(data_array) - seq_length

    def __len__(self):
        return self.end

    def __getitem__(self, idx):
        x_seq = self.data[idx : idx + self.seq_length]       # shape: [seq_length]
        y_val = self.data[idx + self.seq_length]             # single float
        x_seq = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(-1)  # [seq_len, 1]
        y_val = torch.tensor([y_val], dtype=torch.float32)    # [1]
        return x_seq, y_val

# Instantiate dataset
seq_length = 30
full_dataset = TempForecastDataset(temps, seq_length=seq_length)
dataset_size = len(full_dataset)

# train/val/test splits
train_size = int(0.7 * dataset_size)
val_size   = int(0.15 * dataset_size)
test_size  = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [ ]:
'''
My model is a Vanilla RNN that processes a sequence of daily minimum
temperatures over several time steps. Internally, it uses a single hidden layer
with a tanh activation at each time step. I instantiated nn.RNN(input_size=1,
hidden_size=16), meaning the network expects a single feature per time step
(the temperature) and produces a hidden state of dimension 16 after each update.
After processing the final time step, I take the last hidden state and pass it
through a fully connected linear layer without any additional activation
function, yielding a single numeric output. This structure allows the model to
learn short-term dependencies via the tanh-based hidden state transitions, but
it lacks the gating mechanisms of more advanced architectures like LSTM or GRU.
'''

# Define a basic RNN model (tanh-based) with a single hidden layer
class VanillaRNN(nn.Module):
    """
    A simple recurrent neural network using nn.RNN with tanh activation.
    Take the last time step's hidden state for regression.
    """
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.hidden_dim = hidden_dim
        # input_size=1 because we have one feature (temp)
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)  # output a single temperature

    def forward(self, x):
        # x shape: (batch_size, seq_length, 1)
        rnn_out, h_n = self.rnn(x)   # rnn_out: (batch, seq_length, hidden_dim)
        # Take the last time step: rnn_out[:, -1, :]
        last_hidden = rnn_out[:, -1, :]
        out = self.fc(last_hidden)   # shape: (batch_size, 1)
        return out

# Initialize model, define loss/optimizer
vanilla_rnn = VanillaRNN(hidden_dim=16)
criterion = nn.MSELoss()  # We'll measure MSE for training
optimizer = optim.Adam(vanilla_rnn.parameters(), lr=1e-3)

# Helper function for training + validating
def rmse(preds, targets):
    return torch.sqrt(torch.mean((preds - targets)**2))

epochs = 10
train_mse_history = []
val_rmse_history = []

# Training loop
for epoch in range(epochs):
    vanilla_rnn.train()
    train_loss = 0.0
    train_batches = 0

    for x_seq, y_val in train_loader:
        optimizer.zero_grad()
        preds = vanilla_rnn(x_seq)
        loss = criterion(preds, y_val)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_batches += 1

    train_mse = train_loss / train_batches
    train_mse_history.append(train_mse)

    # Validation
    vanilla_rnn.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for x_seq, y_val in val_loader:
            preds = vanilla_rnn(x_seq)
            val_loss += rmse(preds, y_val).item()
            val_batches += 1
    val_loss /= val_batches
    val_rmse_history.append(val_loss)

    print(f"[Epoch {epoch+1}/{epochs}] Train MSE: {train_mse:.4f} | Val RMSE: {val_loss:.4f}")

# Visualize the training MSE and validation RMSE for the vanilla RNN
plt.figure(figsize=(8,4))
epochs_range = range(1, epochs+1)

plt.subplot(1,2,1)
plt.title("Vanilla RNN - Train MSE")
plt.plot(epochs_range, train_mse_history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("MSE")

plt.subplot(1,2,2)
plt.title("Vanilla RNN - Val RMSE")
plt.plot(epochs_range, val_rmse_history, marker='o')
plt.xlabel("Epoch")
plt.ylabel("RMSE")

plt.tight_layout()
plt.show()

# Evaluate on Test set + optionally visualize predictions
vanilla_rnn.eval()
test_rmse_score = 0.0
test_batches = 0

all_preds = []
all_targets = []
with torch.no_grad():
    for x_seq, y_val in test_loader:
        preds = vanilla_rnn(x_seq)
        all_preds.append(preds)
        all_targets.append(y_val)
        test_rmse_score += rmse(preds, y_val).item()
        test_batches += 1

test_rmse_score /= test_batches
print(f"Vanilla RNN Test RMSE: {test_rmse_score:.4f}")

# Flatten predictions for a quick plot: predictions vs. ground truth
all_preds = torch.cat(all_preds).flatten().numpy()
all_targets = torch.cat(all_targets).flatten().numpy()

plt.figure()
plt.title("Vanilla RNN - Test Predictions vs Actual")
plt.plot(all_targets[:100], label="Actual")  # show first 100 for clarity
plt.plot(all_preds[:100], label="Predicted")
plt.xlabel("Test Sample")
plt.ylabel("Temperature")
plt.legend()
plt.show()

## Part 2

In [ ]:
# Define LSTM and GRU variants
class LSTMModel(nn.Module):
    """
    Single-layer LSTM. Take the last time step's output for regression.
    """
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        # last time step: lstm_out[:, -1, :]
        last_hidden = lstm_out[:, -1, :]
        out = self.fc(last_hidden)
        return out

class GRUModel(nn.Module):
    """
    Single-layer GRU. Take the last time step's output for regression.
    """
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        gru_out, h_n = self.gru(x)
        last_hidden = gru_out[:, -1, :]
        out = self.fc(last_hidden)
        return out

# 2.2) We'll train each new model (LSTM, GRU) similarly, comparing to the vanilla RNN
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse_loss_fn = nn.MSELoss()
    train_mse_list = []
    val_rmse_list = []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        batches = 0
        for x_seq, y_val in train_loader:
            optimizer.zero_grad()
            preds = model(x_seq)
            loss = mse_loss_fn(preds, y_val)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            batches += 1
        train_mse = total_train_loss / batches
        train_mse_list.append(train_mse)

        # validation
        model.eval()
        val_rmse_val = 0.0
        val_batches = 0
        with torch.no_grad():
            for x_seq, y_val in val_loader:
                preds = model(x_seq)
                val_rmse_val += rmse(preds, y_val).item()
                val_batches += 1
        val_rmse_val /= val_batches
        val_rmse_list.append(val_rmse_val)

        print(f"Epoch {epoch+1}/{epochs}, Train MSE: {train_mse:.4f}, Val RMSE: {val_rmse_val:.4f}")

    return train_mse_list, val_rmse_list

# 2.3) Instantiate LSTM and GRU
lstm_model = LSTMModel(hidden_dim=16)
gru_model = GRUModel(hidden_dim=16)

print("\n--- Training LSTM ---")
lstm_train_mse, lstm_val_rmse = train_model(lstm_model, train_loader, val_loader, epochs=10)

print("\n--- Training GRU ---")
gru_train_mse, gru_val_rmse = train_model(gru_model, train_loader, val_loader, epochs=10)

# 2.4) Visualize training MSE and validation RMSE for LSTM and GRU
epochs = range(1, 11)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.title("Train MSE (LSTM vs GRU)")
plt.plot(epochs, lstm_train_mse, label="LSTM", marker='o')
plt.plot(epochs, gru_train_mse, label="GRU", marker='o')
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1,2,2)
plt.title("Val RMSE (LSTM vs GRU)")
plt.plot(epochs, lstm_val_rmse, label="LSTM", marker='o')
plt.plot(epochs, gru_val_rmse, label="GRU", marker='o')
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate each on the test set
def evaluate_test(model, test_loader):
    model.eval()
    total_rmse = 0.0
    batches = 0
    with torch.no_grad():
        for x_seq, y_val in test_loader:
            preds = model(x_seq)
            total_rmse += rmse(preds, y_val).item()
            batches += 1
    return total_rmse / batches

lstm_test_rmse = evaluate_test(lstm_model, test_loader)
gru_test_rmse  = evaluate_test(gru_model, test_loader)

print(f"LSTM Test RMSE: {lstm_test_rmse:.4f}")
print(f"GRU  Test RMSE: {gru_test_rmse:.4f}")

## Part 3

Yes, you can technically use a feed-forward network for time-series forecasting by converting the sequence into fixed features (for instance, taking the last \(N\) days of data as a single vector). This means each input to the MLP is just a flattened window of past values. However, a feed-forward network does not inherently capture temporal dependencies, because it treats those \(N\) values as static inputs rather than a sequence. RNN-based models (like LSTM or GRU) can be more effective for time-series tasks, since they maintain a hidden state across time steps and thus better model long-range or dynamic patterns in the data.

# Task 3

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

'''
I chose BERT embeddings because they are context-aware.
In contrast to static embeddings like Word2Vec, GloVe, or FastText,
BERT’s embeddings change based on context—providing more precise
semantic representations. Additionally, even when a word is not found
as a whole, BERT automatically decomposes it into subwords, allowing
us to compute an approximate embedding (e.g., by averaging subword embeddings).
'''

# Load pre-trained BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model_bert = AutoModel.from_pretrained("bert-base-uncased")
model_bert.eval()  # set to evaluation mode

def get_bert_embedding(word, aggregation="mean"):
    """
    Given a word, returns its embedding from BERT.
    If the word is split into subwords, aggregate them (mean or sum).
    """
    # Tokenize the word; add special tokens if needed.
    tokens = tokenizer.tokenize(word)

    # If no tokens are returned, word is OOV.
    if len(tokens) == 0:
        print(f"Word '{word}' is out-of-vocabulary; using random vector as approximation.")
        return torch.randn(model_bert.config.hidden_size)

    # Encode and obtain hidden states
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = model_bert(**inputs)
    # outputs.last_hidden_state shape: [1, sequence_length, hidden_size]
    # Remove batch dimension:
    token_embeddings = outputs.last_hidden_state.squeeze(0)  # shape: [seq_length, hidden_size]

    # BERT always produces subword tokens; aggregate them.
    if aggregation == "mean":
        embedding = token_embeddings.mean(dim=0)
    elif aggregation == "sum":
        embedding = token_embeddings.sum(dim=0)
    else:
        embedding = token_embeddings[0]  # take first token (not recommended)
    return embedding

def dynamic_user_input_embeddings():
    """
    Prompts the user to input two words and prints their embeddings.
    """
    word1 = input("Enter first word: ").strip()
    word2 = input("Enter second word: ").strip()
    emb1 = get_bert_embedding(word1)
    emb2 = get_bert_embedding(word2)
    print(f"Embedding for '{word1}':\n", emb1.numpy()[:10], "...")  # show first 10 dims
    print(f"Embedding for '{word2}':\n", emb2.numpy()[:10], "...")
    return emb1, emb2

dynamic_user_input_embeddings()

In [ ]:
def cosine_similarity(emb1, emb2):
    """Compute cosine similarity between two embeddings."""
    return F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()

def batch_cosine_similarity(word_pairs):
    """
    Given a list of tuples [(word1, word2), ...], compute cosine similarities.
    Returns a list of (word1, word2, similarity).
    """
    similarities = []
    for w1, w2 in word_pairs:
        emb1 = get_bert_embedding(w1)
        emb2 = get_bert_embedding(w2)
        sim = cosine_similarity(emb1, emb2)
        similarities.append((w1, w2, sim))
    return similarities

'''
Cosine similarity is useful in word embedding space because it
measures the cosine of the angle between two vectors—indicating their orientation
(and thus semantic similarity) regardless of their magnitudes.
'''

# Visualization: Plot a 2D scatter plot of embeddings for a list of words using PCA.
def visualize_embeddings(word_list):
    """
    Given a list of words, compute embeddings, reduce dimensions via PCA,
    and plot a 2D scatter plot.
    """
    embeddings = []
    for word in word_list:
        emb = get_bert_embedding(word)
        embeddings.append(emb.numpy())
    embeddings = np.array(embeddings)  # shape: [num_words, hidden_size]

    # Reduce to 2 dimensions using PCA
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(embeddings)

    plt.figure(figsize=(8,6))
    plt.scatter(reduced[:, 0], reduced[:, 1])
    for i, word in enumerate(word_list):
        plt.annotate(word, (reduced[i, 0], reduced[i, 1]))
    plt.title("2D Visualization of BERT Embeddings (PCA)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

words_to_visualize = ["cat", "dog", "lion", "tiger", "table", "chair", "apple", "orange", "bank", "money"]
visualize_embeddings(words_to_visualize)

In [ ]:
'''
Here I designed a custom dissimilarity metric that combines both cosine similarity
and Euclidean distance. The idea is that while cosine similarity focuses on the
orientation (semantic similarity), Euclidean distance reflects the absolute distance
between word vectors. The custom metric is defined as:

    D_custom = alpha * (1 - cosine_similarity) + (1 - alpha) * (normalized Euclidean distance)

where alpha in [0,1] allows the user to toggle between emphasizing cosine similarity
(alpha near 1) and Euclidean distance (alpha near 0). The Euclidean distance is normalized
by dividing by the maximum observed distance among a batch to make the two terms comparable.
'''

def euclidean_distance(emb1, emb2):
    return torch.norm(emb1 - emb2).item()

def custom_dissimilarity(emb1, emb2, alpha=0.5, max_dist=None):
    """
    Custom dissimilarity metric that combines cosine similarity and Euclidean distance.
    - alpha: weight for cosine similarity (1 - cosine similarity gives a dissimilarity measure).
    - (1-alpha): weight for normalized Euclidean distance.
    - max_dist: maximum distance to normalize the Euclidean component. If None, we compute it.
    """
    cos_sim = cosine_similarity(emb1, emb2)  # value in [-1, 1]
    # Transform cosine similarity to a dissimilarity: higher value means more dissimilar.
    cos_dissim = 1 - ((cos_sim + 1) / 2)  # maps cosine similarity [-1,1] to dissimilarity [1,0]
    # Euclidean distance
    euc_dist = euclidean_distance(emb1, emb2)
    # If max_dist is not provided, use a rough estimate (or compute across a sample set).
    if max_dist is None:
        max_dist = 20  # this is arbitrary; in practice, you may compute it over a batch of words.
    norm_euc = euc_dist / max_dist  # normalized to approximately [0, 1]

    # Combine the two measures
    custom_diss = alpha * cos_dissim + (1 - alpha) * norm_euc
    return custom_diss

def batch_custom_dissimilarity(word_pairs, alpha=0.5, max_dist=None):
    results = []
    for w1, w2 in word_pairs:
        emb1 = get_bert_embedding(w1)
        emb2 = get_bert_embedding(w2)
        diss = custom_dissimilarity(emb1, emb2, alpha=alpha, max_dist=max_dist)
        results.append((w1, w2, diss))
    return results

# Visualization: Given a reference word, rank a list of words by similarity/dissimilarity using both metrics.
def plot_word_rankings(ref_word, word_list, alpha=0.5):
    ref_emb = get_bert_embedding(ref_word)
    rankings = []
    for word in word_list:
        emb = get_bert_embedding(word)
        cos_sim = cosine_similarity(ref_emb, emb)
        diss = custom_dissimilarity(ref_emb, emb, alpha=alpha)
        rankings.append((word, cos_sim, diss))
    # Sort by cosine similarity (descending) and by custom dissimilarity (ascending)
    rankings_sorted_cos = sorted(rankings, key=lambda x: x[1], reverse=True)
    rankings_sorted_diss = sorted(rankings, key=lambda x: x[2])

    words_cos = [r[0] for r in rankings_sorted_cos]
    cos_values = [r[1] for r in rankings_sorted_cos]
    words_diss = [r[0] for r in rankings_sorted_diss]
    diss_values = [r[2] for r in rankings_sorted_diss]

    # Plot rankings side by side
    fig, axs = plt.subplots(1, 2, figsize=(14,6))
    axs[0].barh(words_cos, cos_values, color='skyblue')
    axs[0].invert_yaxis()
    axs[0].set_title(f"Cosine Similarity Ranking to '{ref_word}'")
    axs[0].set_xlabel("Cosine Similarity")

    axs[1].barh(words_diss, diss_values, color='salmon')
    axs[1].invert_yaxis()
    axs[1].set_title(f"Custom Dissimilarity Ranking to '{ref_word}' (alpha={alpha})")
    axs[1].set_xlabel("Custom Dissimilarity")
    plt.tight_layout()
    plt.show()

# Ranking visualization:
reference_word = "cat"
comparison_words = ["dog", "lion", "table", "computer", "car", "mouse", "elephant", "cup"]
plot_word_rankings(reference_word, comparison_words, alpha=0.7)

# Heatmap visualization: compare similarity/dissimilarity across multiple word pairs.
def plot_similarity_heatmap(word_list, metric="cosine", alpha=0.5, max_dist=None):
    n = len(word_list)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            emb_i = get_bert_embedding(word_list[i])
            emb_j = get_bert_embedding(word_list[j])
            if metric == "cosine":
                sim = cosine_similarity(emb_i, emb_j)
                mat[i, j] = sim
            elif metric == "custom":
                diss = custom_dissimilarity(emb_i, emb_j, alpha=alpha, max_dist=max_dist)
                mat[i, j] = diss
    plt.figure(figsize=(8,6))
    sns.heatmap(mat, xticklabels=word_list, yticklabels=word_list, annot=True, cmap="viridis")
    plt.title(f"Heatmap of {metric.capitalize()} Metric")
    plt.show()

# Heatmap for cosine similarity
plot_similarity_heatmap(["cat", "dog", "lion", "table", "car"], metric="cosine")
# Heatmap for custom dissimilarity
plot_similarity_heatmap(["cat", "dog", "lion", "table", "car"], metric="custom", alpha=0.7, max_dist=20)
